# Runge-Kutta: Four Slopes, and What They Buy

**Time:** about 15 to 20 minutes.
**You need:** nothing but this page. New to notebooks? Do `00-start-here.ipynb` first.

This activity goes with Trench section 3.3, and it comes after the by-hand work in your packet.
You will not do any arithmetic here. The packet was for learning how to take a step. This is for
seeing what those steps are doing.

Think about estimating how far you will drive in the next hour. Euler's method glances at the
speedometer once and assumes it holds for the whole hour. If you could also check it halfway
through, check it there a second time, and peek at the end, your estimate would be much better.
That is RK4. For the equation $y'=f(t,y)$ with step $h$, it asks for four slopes:

$$
\begin{aligned}
k_1&=f(t_i,\;y_i) &&\text{slope where you stand}\\
k_2&=f\big(t_i+\tfrac h2,\;y_i+\tfrac h2k_1\big) &&\text{halfway, using }k_1\\
k_3&=f\big(t_i+\tfrac h2,\;y_i+\tfrac h2k_2\big) &&\text{halfway again, using }k_2\\
k_4&=f\big(t_i+h,\;y_i+hk_3\big) &&\text{the far end, using }k_3
\end{aligned}
$$

and then steps with a weighted average of them:

$$y_{i+1}=y_i+\frac h6\big(k_1+2k_2+2k_3+k_4\big).$$

> **The main idea:** slopes spent inside a step buy far more accuracy than slopes spent on more
> steps, and when RK4 goes wrong it may give no warning, so you check it by cutting $h$ in half.

Each section asks you to predict before you run anything. Commit to an answer first, on paper or
out loud. Being wrong and seeing why is the point.

## Run this first

The two cells below set up the drawing tools. You never need to read or change them. Run them
once each (**Shift + Enter**) and give them a moment on the first run.

In [ ]:
%pip install -q ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def euler_step(f, t, y, h):
    'One Euler step: one slope.'
    return y + h * f(t, y)


def improved_step(f, t, y, h):
    'One improved Euler step: two slopes.'
    k1 = f(t, y)
    k2 = f(t + h, y + h*k1)
    return y + h * (k1 + k2) / 2


def rk4_step(f, t, y, h):
    'One RK4 step: four slopes.'
    k1 = f(t, y)
    k2 = f(t + h/2, y + h*k1/2)
    k3 = f(t + h/2, y + h*k2/2)
    k4 = f(t + h, y + h*k3)
    return y + h * (k1 + 2*k2 + 2*k3 + k4) / 6


def march(step, f, t0, y0, h, n):
    'Take n steps of size h with the given method. Returns (ts, ys).'
    ts, ys = [t0], [y0]
    t, y = t0, y0
    for i in range(1, n + 1):
        y = step(f, t, y, h)
        t = t0 + i * h
        ts.append(t)
        ys.append(y)
    return np.array(ts), np.array(ys)


def drug(t, C):
    'The rate of change of the drug concentration.'
    return -2 * C / (1 + C)


def drug_truth(t):
    'True C(t), from the implicit solution ln C + C = 4 + ln 4 - 2t, solved by Newton.'
    target = 4 + np.log(4) - 2 * t
    C = 2.0
    for _ in range(50):
        C = C - (np.log(C) + C - target) / (1 / C + 1)
    return C


LOOK = {'Euler':          dict(color='C0', linestyle=':',  marker='o'),
        'improved Euler': dict(color='C1', linestyle='-.', marker='s'),
        'RK4':            dict(color='C2', linestyle='-',  marker='^')}


def segment(ax, t, y, k, **style):
    'Draw a short segment of slope k through (t, y), the same length on screen for any slope.'
    across, up = 8 / 2.4, 5 / 8.5
    dt = 0.42 / np.sqrt(across**2 + (k * up)**2) * across / 2
    ax.plot([t - dt, t + dt], [y - k*dt, y + k*dt], **style)


def four_slopes_picture(h):
    'One RK4 step on y prime = y from (0, 1), with each slope drawn where it is measured.'
    k1 = 1.0
    p2 = 1 + h/2 * k1
    k2 = p2
    p3 = 1 + h/2 * k2
    k3 = p3
    p4 = 1 + h * k3
    k4 = p4
    land = 1 + h/6 * (k1 + 2*k2 + 2*k3 + k4)
    truth = np.exp(h)

    fig, ax = plt.subplots(figsize=(8, 5))
    tt = np.linspace(0, 2.2, 300)
    ax.plot(tt, np.exp(tt), color='C3', linestyle='--', linewidth=2, label='true solution')

    probes = [(0, 1, k1, 'C0', 'o', f'k1 = {k1:.2f}, where you stand'),
              (h/2, p2, k2, 'C1', 's', f'k2 = {k2:.2f}, halfway, using k1'),
              (h/2, p3, k3, 'C4', 'D', f'k3 = {k3:.2f}, halfway again, using k2'),
              (h, p4, k4, 'C5', 'v', f'k4 = {k4:.2f}, the far end, using k3')]
    for (t, y, k, c, m, label) in probes:
        ax.plot([0, t], [1, y], color=c, linewidth=0.9, linestyle=':')
        segment(ax, t, y, k, color=c, linewidth=3.5)
        ax.plot([t], [y], color=c, marker=m, markersize=9, linestyle='none', label=label,
                markeredgecolor='black', markeredgewidth=0.6, zorder=6)

    ax.plot([0, h], [1, land], color='C2', linewidth=2.2,
            label=f'RK4 step, lands at {land:.4f}')
    ax.plot([h], [land], color='C2', marker='*', markersize=24, linestyle='none',
            markeredgecolor='black', markeredgewidth=0.6, zorder=5)

    ax.set_xlim(-0.1, 2.3)
    ax.set_ylim(0, 8.5)
    ax.set_xlabel('t')
    ax.set_ylabel('y')
    ax.set_title(f"y' = y, y(0) = 1: one RK4 step with h = {h:.2f}, "
                 f"lands at {land:.4f}, truth {truth:.4f}", fontsize=11)
    ax.grid(alpha=0.25)
    ax.legend(loc='upper left', framealpha=0.9, fontsize=9)
    plt.show()
    plt.close(fig)


def budget_bars(budget):
    'Spend the same number of slopes three ways on the drug problem and score each one.'
    truth = drug_truth(1.0)
    plans = [('Euler', euler_step, budget), ('improved Euler', improved_step, budget // 2),
             ('RK4', rk4_step, budget // 4)]

    fig, ax = plt.subplots(figsize=(8, 5))
    for i, (name, step, n) in enumerate(plans):
        ts, ys = march(step, drug, 0.0, 4.0, 1.0 / n, n)
        err = abs(ys[-1] - truth)
        score = -np.log10(err)
        ax.bar(i, score, width=0.6, color=LOOK[name]['color'], edgecolor='black',
               hatch=['', '//', 'xx'][i])
        ax.text(i, score + 0.25, f'C(1) = {ys[-1]:.6f}\nerror {err:.2g}\nscore {score:.1f}',
                ha='center', va='bottom', fontsize=10)

    ax.set_xticks(range(3))
    ax.set_xticklabels([f'{name}\n{n} step{"s" if n > 1 else ""} of h = {1/n:.4g}'
                        for name, step, n in plans])
    ax.set_ylim(0, 14.5)
    ax.set_ylabel('decimal places of accuracy')
    ax.set_title(f"C' = -2C/(1+C), C(0) = 4: estimating C(1) on a budget of {budget} slopes")
    ax.grid(axis='y', alpha=0.25)
    plt.show()
    plt.close(fig)


def stability_picture(h):
    'All three methods on y prime = -5y, y(0) = 2, with a shared step size h.'
    f = lambda t, y: -5 * y
    z = -5 * h
    mult = {'Euler': 1 + z,
            'improved Euler': 1 + z + z**2/2,
            'RK4': 1 + z + z**2/2 + z**3/6 + z**4/24}
    n = int(np.ceil(4 / h))

    fig, ax = plt.subplots(figsize=(8, 5))
    tt = np.linspace(0, 4, 400)
    ax.plot(tt, 2 * np.exp(-5 * tt), color='C3', linestyle='--', linewidth=2.5,
            label='true solution')
    for name, step in [('Euler', euler_step), ('improved Euler', improved_step),
                       ('RK4', rk4_step)]:
        ts, ys = march(step, f, 0.0, 2.0, h, n)
        ax.plot(ts, ys, linewidth=2, markersize=5, label=name, **LOOK[name])

    ax.axhline(0, color='0.6', linewidth=0.8)
    ax.set_xlim(0, 4)
    ax.set_ylim(-4, 4)
    ax.set_xlabel('t')
    ax.set_ylabel('y')
    ax.set_title(f"y' = -5y, y(0) = 2 with h = {h:.2f}\n"
                 f"each step multiplies by: Euler {mult['Euler']:.2f}, "
                 f"improved Euler {mult['improved Euler']:.2f}, RK4 {mult['RK4']:.2f}, "
                 f"truth {np.exp(z):.2f}", fontsize=11)
    ax.grid(alpha=0.25)
    ax.legend(loc='upper right', framealpha=0.9)
    plt.show()
    plt.close(fig)


print('Ready.')

## 1. Where the four slopes live

Go back to the simplest equation in the course:

$$y'=y, \qquad y(0)=1.$$

Read it as a sentence: *the slope equals the height*. That makes it the best place to look at RK4,
because on this equation you can read a slope straight off how high a point sits.

Picture one RK4 step of size $h=1$. In `02-eulers-method.ipynb`, one Euler step of this size landed
at $2$, and the truth is about $2.718$.

Before running anything, predict:

1. Which of the four slopes will be the steepest, and why?
2. $k_2$ and $k_3$ are both measured halfway across. Will the point for $k_3$ sit above or below
   the point for $k_2$?
3. Will the true curve pass above both halfway points, below both, or between them?

Now run the cell at $h=1$ and check your answers before you touch the slider.

In [ ]:
from ipywidgets import interact, FloatSlider

def explore_four_slopes(h=1.0):
    four_slopes_picture(h)

interact(
    explore_four_slopes,
    h=FloatSlider(value=1.0, min=0.25, max=2.0, step=0.25,
                  description='h', continuous_update=False)
);

### What the picture shows

Each thick short segment is one of the four slopes, drawn at the exact point where RK4 measures it.
The thin dotted lines show the path from the start to each of those points. The green line is the
step RK4 actually takes, and the star is where it lands. Notice that the green line matches none
of the four short segments. It is a blend of all of them.

<details>
<summary>Check yourself after you have written something</summary>

The slopes are $1$, $1.5$, $1.75$, and $2.75$, so $k_4$ is the steepest: it is measured at the
highest point, and here the slope <em>is</em> the height. The $k_3$ point sits above the $k_2$ point
because $k_3$ walked out along the steeper slope $k_2$ instead of $k_1$. The true curve passes
between them, at about $1.65$. RK4 lands at $2.7083$, and Euler landed at $2$.
</details>

### What changed, and what stayed the same?

Now drag $h$ to both ends on purpose, $0.25$ and $2$.

**In your notes:** at $h=0.25$, what has happened to the four points? At $h=2$, does the true curve
still pass between the two halfway points? And how far is the star from the true curve at $h=2$,
compared with the $3$ that a single Euler step of that size would give you?

<details>
<summary>Check yourself after you have written something</summary>

At $h=0.25$ the four points pile up and the four slopes nearly agree, so there is little left to
correct. At $h=2$ the points spread far apart, and the true curve still runs between the two
halfway points. One halfway reading comes in low and the other comes in high. Together they make a
good estimate of the slope at the middle of the step, and the middle is what counts double in the
average. That is how RK4 lands at $7$ against a truth of about $7.39$, while Euler lands at $3$.
</details>

You just explained why RK4 works without computing a single $k$.

## 2. Same slopes, three ways to spend them

Back to the drug from your packet. Many drugs are cleared by enzymes that can only work so fast:

$$C'=\frac{-2C}{1+C}, \qquad C(0)=4.$$

Read it as a sentence: *the more drug there is, the faster it clears, but the rate levels off
because the enzymes are maxed out*. We want $C(1)$, the concentration after one hour.

Treat each slope you ask for as one unit of work. A budget of $4$ slopes buys four Euler steps, or
two improved Euler steps, or one RK4 step. A budget of $8$ buys twice as many steps of each.

The bars below score each method in decimal places of accuracy: an error of $0.01$ scores $2$, and
an error of $0.001$ scores $3$. We can score them because this equation separates, which gives us
the true $C(1)$ to compare against.

Before running anything, predict:

1. Each notch of the slider doubles the budget. Which bar gains the most per notch? Roughly how
   many times faster than Euler's bar do you think it climbs?
2. How many slopes does Euler need to match what RK4 does with only $4$? Write down an actual
   number. Is it closer to $10$, $100$, or $1000$?

Now run the cell and push the slider all the way to both ends.

In [ ]:
from ipywidgets import interact, SelectionSlider

def explore_budget(budget=4):
    budget_bars(budget)

interact(
    explore_budget,
    budget=SelectionSlider(options=[4, 8, 16, 32, 64, 128, 256], value=4,
                           description='slopes', continuous_update=False)
);

### What changed, and what stayed the same?

All three bars grow with every notch, and the order never changes. What to watch is the size of
each jump, because that is the exchange rate each method offers you.

**In your notes,** fill these in from the scores on the bars:

- Each doubling of the budget adds about ________ to Euler's score.
- Each doubling adds about ________ to improved Euler's score.
- Each doubling adds about ________ to RK4's score.

<details>
<summary>Check yourself after you have written something</summary>

About $0.3$, then $0.6$, then $1.2$. RK4's bar climbs four times as fast as Euler's, and twice as
fast as improved Euler's. At $256$ slopes the scores are about $3.5$, $6.1$, and $11.2$. Spending
the same work, RK4 is accurate to eleven decimal places and Euler is still working on its fourth.
</details>

Now your second prediction. This cell gives Euler more and more slopes until it finally matches
what RK4 did with four.

In [ ]:
target = abs(march(rk4_step, drug, 0.0, 4.0, 1.0, 1)[1][-1] - drug_truth(1.0))

n = 4
while abs(march(euler_step, drug, 0.0, 4.0, 1.0 / n, n)[1][-1] - drug_truth(1.0)) > target:
    n = n + 1

print(f'RK4 with 4 slopes has error {target:.6f}')
print(f'Euler first gets that close at {n} slopes')
print(f'That is about {n / 4:.0f} times the work.')

<details>
<summary>Check yourself after you have compared it to your guess</summary>

Euler needs $761$ slopes, about $190$ times the work, to match one RK4 step. Each of those slopes
is computed correctly. Euler's trouble is that it only ever looks at the start of each step.
</details>

You compared three methods with nothing but a budget and a score, and you never had to look at a
formula for $C(t)$. The same exchange rates carry over to equations you could never solve, which is
why nearly every serious solver is built on this idea.

## 3. Halve the step

The bars gave you the exchange rates. Here is the same fact in the form Trench uses, with the
table from `02-eulers-method.ipynb` extended to all three methods.

We use $y'=y$, $y(0)=1$, marching to $t=3$, where the true value is $e^3$. Each row cuts $h$ in
half. The columns marked *drop* show how many times smaller the error got compared with the row
above.

Before running, predict:

1. What number will each *drop* column settle near? Write down three numbers.
2. By the last row, will they have reached those numbers exactly, or still be on the way?

In [ ]:
f = lambda t, y: y
true = np.exp(3)

print(f"y' = y, y(0) = 1, marching to t = 3.   True value: e^3 = {true:.6f}")
print()
print(f"{'h':>9} {'Euler err':>11} {'drop':>7} {'improved err':>14} {'drop':>7} "
      f"{'RK4 err':>11} {'drop':>7}")
print('-' * 72)

prev = None
for h in [0.5, 0.25, 0.125, 0.0625, 0.03125]:
    n = round(3 / h)
    errs = [abs(true - march(step, f, 0.0, 1.0, h, n)[1][-1])
            for step in (euler_step, improved_step, rk4_step)]
    drops = [f'{p/e:.2f}' for p, e in zip(prev, errs)] if prev else ['-', '-', '-']
    print(f'{h:>9.5f} {errs[0]:>11.5f} {drops[0]:>7} {errs[1]:>14.5f} {drops[1]:>7} '
          f'{errs[2]:>11.7f} {drops[2]:>7}')
    prev = errs

### Reading the table

Look down each *drop* column. They are climbing toward their targets without quite reaching them,
which is what an exchange rate looks like in real numbers. Then look at the last row and compare
the sizes of the three errors.

**In your notes:** suppose you need an error $10{,}000$ times smaller than the one you have. How
much would you have to shrink $h$ for Euler? For improved Euler? For RK4?

<details>
<summary>Check yourself after you have written something</summary>

The columns head for $2$, $4$, and $16$, and at the last row they have reached about $1.92$,
$3.90$, and $15.59$. Those targets are $2^1$, $2^2$, and $2^4$, which is what Trench means by
$O(h)$, $O(h^2)$, and $O(h^4)$.

So error follows $h$, or $h^2$, or $h^4$. To make it $10{,}000$ times smaller, Euler has to shrink
$h$ by $10{,}000$, improved Euler by $100$, and RK4 by only $10$.
</details>

## 4. When RK4 breaks, it may not warn you

So far every comparison has been about accuracy. In `02-eulers-method.ipynb` you also saw a sharper
kind of failure: past a certain step size, Euler stopped being a bit off and started zigzagging
wildly. That zigzag was ugly, and it was also useful, because nobody could mistake it for an
answer.

Take a system that settles fast, like a drug that clears quickly:

$$y'=-5y, \qquad y(0)=2.$$

Read it as a sentence: *the rate of change is five times the current amount, pointing back toward
zero*. The truth is $y=2e^{-5t}$, which drops to zero and stays there.

In your packet, at $h=0.4$, Euler bounced between $2$ and $-2$, improved Euler froze at $2$, and
RK4 decayed. So RK4 looks safe. Before running anything, predict:

1. Is there a step size where RK4 stops decaying too, or is it safe for any $h$?
2. If it does break, will it zigzag the way Euler does?

Now run the cell and drag $h$ slowly from left to right, all the way to the end. Then find the
exact spot where each method stops decaying. A curve that leaves the picture is gone for good, and
the numbers in the title keep telling you what is happening.

In [ ]:
from ipywidgets import interact, FloatSlider

def explore_stability(h=0.1):
    stability_picture(h)

interact(
    explore_stability,
    h=FloatSlider(value=0.1, min=0.05, max=0.7, step=0.01,
                  description='h', continuous_update=False)
);

### What changed, and what stayed the same?

On this equation every step multiplies by a fixed number, the same idea as the growth factor in
`02`. Those numbers are in the title. If the multiplier is between $-1$ and $1$, the method decays.
Past that, it grows.

**In your notes,** fill these in from the slider:

- Euler stops decaying at $h=$ ________.
- Improved Euler stops decaying at $h=$ ________.
- RK4 stops decaying between $h=$ ________ and $h=$ ________.
- Which of the three multipliers ever goes negative?

<details>
<summary>Check yourself after you have written something</summary>

Euler and improved Euler both stop at $h=0.4$. RK4 stops between $0.55$ and $0.56$. So RK4 does have
a limit, and more slopes moved it further out without removing it.

Only Euler's multiplier ever goes negative, and a negative multiplier is what makes a zigzag.
Improved Euler's multiplier never drops below $0.5$, and RK4's never drops below about $0.27$. So
past their limits these two rise smoothly, and on this equation they never flip sign. At $h=0.6$,
RK4 draws a clean, confident curve climbing away from a truth that is sitting at zero.
</details>

That is the catch. Euler's failure was loud. On this equation, RK4's is quiet: its curve looks like
something a real system could do.

Now go back to $h=0.5$. RK4 is officially decaying there, with a multiplier of $0.65$. The truth has
a multiplier of $0.08$. Staying under the limit keeps the curve from blowing up. It does not make
the curve right.

So the shape of the curve can't be your test. Equations with more going on can fail in messier
ways, like the bouncing $h=0.2$ column of the fish problem in your packet. You need a test that
works whether the failure is loud or quiet, and you already know one. You used it in that same
problem.

## 5. Final challenge: read this one cold

No scaffolding here.

$$y'=t^2+y^2, \qquad y(0)=1.$$

Read it as a sentence: *the slope is the squared distance from the origin*. The further out a
solution gets, the steeper it climbs. This equation doesn't separate, and no method in this course
gives you a formula for it. You have no truth to compare against, so all you have is what the
computer tells you.

The next cell runs RK4 at three step sizes and prints the results side by side.

Before running it, write down in your notes:

1. The rule you will use to decide which rows to believe.
2. What you would expect to see if a row can't be trusted.

Then run it and answer: what is your best report for $y(0.8)$, and how many digits would you stand
behind? What about $y(0.9)$? What about $y(1)$?

In [ ]:
f = lambda t, y: t*t + y*y
sizes = [0.1, 0.05, 0.025]

runs = [march(rk4_step, f, 0.0, 1.0, h, round(1.0 / h)) for h in sizes]

print("y' = t^2 + y^2, y(0) = 1, RK4 at three step sizes")
print()
print(f"{'t':>5} {'h = 0.1':>18} {'h = 0.05':>18} {'h = 0.025':>18}")
print('-' * 62)
for t in [0.2, 0.4, 0.6, 0.8, 0.9, 1.0]:
    row = [ys[round(t / h)] for h, (ts, ys) in zip(sizes, runs)]
    cells = [f'{v:>18.4f}' if v < 1e7 else f'{v:>18.3g}' for v in row]
    print(f'{t:>5.1f} ' + ' '.join(cells))

<details>
<summary>Check yourself once you have written your answers down</summary>

The rule: believe the digits that stay put when $h$ is cut in half. If a digit keeps changing, you
can't stand behind it.

Through $t=0.6$ the columns agree to about three decimal places. At $t=0.8$ the last two columns
agree on $5.85$, so report that. At $t=0.9$ the last two columns give $14.27$ and $14.30$, so you could stand behind
about $14.3$ and no more. At $t=1$ the columns read $735$, then about $176{,}000$, then about
$10^{21}$. Halving $h$ is making the answer wildly bigger, so there is no value to report.

Here is why. The true solution blows up near $t=0.97$, so at $t=1$ there is nothing for RK4 to
approximate. This is the same trap as $y'=y^2$ in `02-eulers-method.ipynb`, the number that means
nothing. The difference is that this time you caught it, with no formula to compare against.
</details>

## Wrap-up

RK4 is one idea: ask for the slope several times inside a step, and step with the blend. On
$y'=y$ you watched the two halfway readings land on either side of the truth, which is why the
blend came out so well.

You saw what that buys. On the same budget RK4 beat Euler by a wide margin, and halving $h$ cuts
its error by about $16$ where Euler's is cut by $2$. It also pushed the stability limit further
out, though it did not remove it.

You also saw what it costs. RK4 can fail with no zigzag to warn you, and it will print a number
for a solution that no longer exists. Neither failure shows up in the method, so the check has to
come from you. Run it, cut $h$ in half, run it again, and keep only the digits that held still.

That check needs no formula, which makes it the most useful thing here. Most equations that
describe real systems have no formula, and this check works on all of them. When you do the RK4
spreadsheet problems for this section, that check is one more run with a smaller $h$.

### Exit question

In two or three sentences, explain this in your own words:

> RK4 can be badly wrong without looking wrong.

Then say exactly what you would do to find out whether an RK4 answer deserves to be trusted.